In [1]:
!pwd

/DATA/lepetit/ls-test/Pipe2/3-AD4T


In [3]:
import os
from pathlib import Path
import numpy as np
import pandas as pd

from metrics2 import DatasetEvaluator


# ==============================================================================
# 1. ORIENTATION DES MÉTRIQUES (Sens du tri pour le ranking : True = plus petit = meilleur)
# ==============================================================================
LOWER_IS_BETTER_CONT = {
    "MSE": True, "MAE": True, "MAPE": True, "SMAPE": True,
    "JS": True, "KL": True, "MMD": True, "R2": False
}

LOWER_IS_BETTER_CAT = {
    "accuracy_globale": False, "accuracy_minoritaire": False,
    "macro_f1_arith": False, "weighted_f1": False,
    "fsg_macro": False, "fsg_weighted": False,
    "gmean_strict": False, "gmean_smoothed": False,
    "macro_auc_roc": False, "weighted_auc_roc": False,
    "auc_pr_macro": False
}

COL_NAMES = ["FC", "PAS", "PAM", "PAD", "Temp", "SpO2", "FR", "event_code"]


# ==============================================================================
# 2. PARCOURS ET ÉVALUATION
# ==============================================================================
def collect_and_evaluate(root_dir: str):
    """
    Parcourt récursivement root_dir à la recherche des dossiers contenant
    real_data.npy et gen_data.npy, puis calcule les métriques.
    """
    root_path = Path(root_dir).resolve()
    if not root_path.exists():
        raise FileNotFoundError(f"Chemin racine introuvable : {root_path}")

    cont_records = []
    cat_records = []

    print(f"Recherche récursive dans : {root_path} ...\n")

    for dirpath, _, filenames in os.walk(root_path):
        if "real_data_dit.npy" in filenames and "gen_data_dit.npy" in filenames:
            folder = Path(dirpath)
            config_name = str(folder.relative_to(root_path))
            print(f"-> Traitement de : {config_name}")

            real_data = np.load(folder / "real_data_dit.npy")
            gen_data = np.load(folder / "gen_data_dit.npy")

            # Nom de dossier sûr pour Windows / POSIX
            safe_name = config_name.replace(os.sep, "_").replace("/", "_").replace("\\", "_")

            try:
                evaluator = DatasetEvaluator(
                    real_data,
                    gen_data,
                    col_names=COL_NAMES,
                    path_dir=f"checkpoints/{safe_name}",
                )
                res_cont, res_cat = evaluator.run_full_analysis()
            except Exception as e:
                print(f"   [ERREUR] Évaluation impossible pour {config_name} : {e}")
                continue

            # --- Variables continues ---
            if isinstance(res_cont, dict):
                records_list = res_cont.get("records", [])
            else:
                records_list = res_cont

            for r in records_list:
                if not isinstance(r, dict):
                    continue
                item = r.copy()
                item["config"] = config_name
                cont_records.append(item)

            # --- Variable catégorielle ---
            if isinstance(res_cat, dict):
                item = res_cat.copy()
                item["config"] = config_name
                cat_records.append(item)

    df_cont = pd.DataFrame(cont_records)
    df_cat = pd.DataFrame(cat_records)
    return df_cont, df_cat


# ==============================================================================
# 3. CALCUL DES RANKINGS DÉTAILLÉS ET GLOBAUX
# ==============================================================================
def _mean_or_nan(df: pd.DataFrame, cols: list) -> pd.Series:
    """Moyenne ligne par ligne en tolérant une liste de colonnes vide."""
    if not cols:
        return pd.Series(np.nan, index=df.index)
    return df[cols].mean(axis=1)


def compute_all_rankings(df_cont: pd.DataFrame, df_cat: pd.DataFrame):
    """
    Calcule :
    - Le rang par variable et par métrique
    - Le leaderboard par variable (meilleure config pour chaque variable)
    - Le leaderboard catégoriel
    - Le leaderboard global consolidé
    """
    # --------------------------------------------------------------------------
    # A. Ranking Variables Continues
    # --------------------------------------------------------------------------
    df_cont_ranked = df_cont.copy()
    cont_metrics = [c for c in df_cont.columns if c not in ["config", "variable"]]

    for m in cont_metrics:
        asc = LOWER_IS_BETTER_CONT.get(m, True)
        df_cont_ranked[f"{m}_rank"] = (
            df_cont_ranked.groupby("variable")[m]
            .rank(ascending=asc, method="min")
        )

    pointwise_cols = [f"{m}_rank" for m in ["MSE", "MAE", "MAPE", "SMAPE", "R2"]
                      if f"{m}_rank" in df_cont_ranked]
    distrib_cols   = [f"{m}_rank" for m in ["JS", "KL", "MMD"]
                      if f"{m}_rank" in df_cont_ranked]

    df_cont_ranked["rank_pointwise"]    = _mean_or_nan(df_cont_ranked, pointwise_cols)
    df_cont_ranked["rank_distribution"] = _mean_or_nan(df_cont_ranked, distrib_cols)
    df_cont_ranked["var_score"] = (
        df_cont_ranked["rank_pointwise"] + df_cont_ranked["rank_distribution"]
    ) / 2

    # Pivot robuste (tolère les doublons de (config, variable))
    best_per_var_pivot = df_cont_ranked.pivot_table(
        index="config", columns="variable", values="var_score", aggfunc="mean"
    )

    best_config_summary = []
    for var in best_per_var_pivot.columns:
        col = best_per_var_pivot[var].dropna()
        if col.empty:
            continue
        best_cfg = col.idxmin()
        best_rank = col.loc[best_cfg]
        best_config_summary.append({
            "Variable": var,
            "Meilleure_Configuration": best_cfg,
            "Rang_Moyen": round(best_rank, 2),
        })
    df_best_by_var = pd.DataFrame(best_config_summary)

    cont_global_score = (
        df_cont_ranked.groupby("config")["var_score"]
        .mean()
        .reset_index()
        .rename(columns={"var_score": "score_continuous"})
    )

    # --------------------------------------------------------------------------
    # B. Ranking Variable Catégorielle
    # --------------------------------------------------------------------------
    df_cat_ranked = df_cat.copy()
    cat_metrics = [c for c in df_cat.columns if c != "config"]

    for m in cat_metrics:
        asc = LOWER_IS_BETTER_CAT.get(m, False)
        df_cat_ranked[f"{m}_rank"] = df_cat_ranked[m].rank(ascending=asc, method="min")

    imbalanced_cols = [f"{m}_rank" for m in
                       ["accuracy_minoritaire", "macro_f1_arith", "fsg_macro",
                        "gmean_smoothed", "auc_pr_macro"]
                       if f"{m}_rank" in df_cat_ranked]
    global_cols = [f"{m}_rank" for m in
                   ["accuracy_globale", "weighted_f1", "fsg_weighted", "weighted_auc_roc"]
                   if f"{m}_rank" in df_cat_ranked]

    df_cat_ranked["rank_imbalanced"]  = _mean_or_nan(df_cat_ranked, imbalanced_cols)
    df_cat_ranked["rank_global_cat"]  = _mean_or_nan(df_cat_ranked, global_cols)
    df_cat_ranked["score_categorical"] = (
        df_cat_ranked["rank_imbalanced"] + df_cat_ranked["rank_global_cat"]
    ) / 2

    cat_global_score = df_cat_ranked[["config", "score_categorical"]].copy()

    # --------------------------------------------------------------------------
    # C. Leaderboard Global Consolidé
    # --------------------------------------------------------------------------
    leaderboard = pd.merge(cont_global_score, cat_global_score, on="config", how="outer")
    leaderboard["global_rank_score"] = (
        leaderboard[["score_continuous", "score_categorical"]].mean(axis=1)
    )
    leaderboard["Rang_Final"] = (
        leaderboard["global_rank_score"].rank(ascending=True, method="min").astype("Int64")
    )
    leaderboard = leaderboard.sort_values("Rang_Final", na_position="last").reset_index(drop=True)

    return df_cont_ranked, best_per_var_pivot, df_best_by_var, df_cat_ranked, leaderboard


# ==============================================================================
# 4. EXPORT ET VISUALISATION
# ==============================================================================
def save_results(df_cont_ranked,
                 best_per_var_pivot,
                 df_best_by_var,
                 df_cat_ranked,
                 leaderboard,
                 output_dir="results_benchmark"):
    """
    Exporte l'ensemble des résultats et rankings sous forme de fichiers CSV.
    """
    out_path = Path(output_dir)
    out_path.mkdir(parents=True, exist_ok=True)

    leaderboard.to_csv(out_path / "leaderboard_global.csv", index=False)
    df_best_by_var.to_csv(out_path / "meilleure_par_variable.csv", index=False)
    best_per_var_pivot.to_csv(out_path / "rangs_variables_pivot.csv", index=True)
    df_cont_ranked.to_csv(out_path / "details_continues.csv", index=False)
    df_cat_ranked.to_csv(out_path / "details_categorielle.csv", index=False)

    print(f"\n Tous les fichiers CSV ont été enregistrés dans : {out_path.resolve()}")
    print("\n" + "=" * 80)
    print("TOP 5 DES MEILLEURES CONFIGURATIONS GLOBALES :")
    print("=" * 80)
    print(
        leaderboard[["Rang_Final", "config", "score_continuous",
                     "score_categorical", "global_rank_score"]]
        .head(5).to_string(index=False)
    )
    print("\n" + "=" * 80)
    print("MEILLEURE CONFIGURATION PAR VARIABLE :")
    print("=" * 80)
    print(df_best_by_var.to_string(index=False))


# ==============================================================================
# 5. POINT D'ENTRÉE
# ==============================================================================
def main():
    # 1. Parcours récursif et calcul
    df_cont, df_cat = collect_and_evaluate("../3-AD4T")

    if df_cont.empty or df_cat.empty:
        raise RuntimeError("Aucune donnée trouvée : vérifiez le chemin racine ../results")

    # 2. Calcul des rankings
    (
        df_cont_ranked,
        best_per_var_pivot,
        df_best_by_var,
        df_cat_ranked,
        leaderboard,
    ) = compute_all_rankings(df_cont, df_cat)

    # 3. Export
    save_results(
        df_cont_ranked,
        best_per_var_pivot,
        df_best_by_var,
        df_cat_ranked,
        leaderboard,
        output_dir="resultats_benchmark",
    )


if __name__ == "__main__":
    main()

Recherche récursive dans : /DATA/lepetit/ls-test/Pipe2/3-AD4T ...

-> Traitement de : checkpoints_pred16_hist32_vae_nl3_nh_vae8_f16_bs64_dr0.5_cce0.15_dit_hd240_d9_nh_dit8
                             RAPPORT D'ÉVALUATION                                   

=== ÉVALUATION DES VARIABLES CONTINUES ===

---------------------------------------------------------------------------------------------------
Variable   | MSE       | MAE       | MAPE (%)  | SMAPE (%)  | R²      | JS      | KL      | MMD    
---------------------------------------------------------------------------------------------------
FC         | 127.6978  | 5.6906    | 6.05      | 5.85       | 0.832   | 0.1041  | 0.1875  | 0.0012 
PAS        | 112.1794  | 6.1732    | 5.38      | 5.19       | 0.893   | 0.1110  | 0.1029  | 0.0011 
PAM        | 40.4627   | 3.3318    | 4.30      | 4.18       | 0.891   | 0.1233  | 0.1935  | 0.0017 
PAD        | 25.4001   | 2.5481    | 4.41      | 4.24       | 0.880   | 0.1364  | 0.2133  | 0.0016

In [ ]:
import pandas as pd

cat = pd.read_csv("resultats_benchmark/details_categorielle.csv", sep=",")
cont = pd.read_csv("resultats_benchmark/details_continues.csv", sep=",")

In [ ]:
cat.sort_values(by='score_categorical')[['config','score_categorical','accuracy_globale','accuracy_minoritaire']]

,config,score_categorical,accuracy_globale,accuracy_minoritaire
11,checkpoints_pred16_hist32_VAE_nl3_nh8_f16_bs64...,1.000,0.931813,0.785012
13,checkpoints_pred16_hist32_VAE_nl2_nh8_f16_bs12...,2.475,0.906312,0.684521
16,checkpoints_pred16_hist32_VAE_nl4_nh8_f16_bs12...,2.525,0.909188,0.693857
9,checkpoints_pred16_hist32_VAE_nl3_nh8_f16_bs64...,4.000,0.793312,0.272973
3,checkpoints_pred16_hist32_VAE_nl3_nh4_f16_bs64...,5.675,0.776250,0.191155
19,checkpoints_pred16_hist32_VAE_nl3_nh8_f16_bs64...,5.750,0.777250,0.212776
12,checkpoints_pred16_hist32_VAE_nl3_nh8_f16_bs64...,6.575,0.757375,0.130713
14,checkpoints_pred16_hist32_VAE_nl4_nh8_f16_bs12...,8.350,0.740000,0.096806
0,checkpoints_pred16_hist32_VAE_nl2_nh8_f16_bs12...,9.500,0.747938,0.082310
2,checkpoints_pred16_hist32_VAE_nl2_nh8_f16_bs12...,11.400,0.727313,0.063882


In [ ]:
import pandas as pd
from pathlib import Path

# ------------------------------------------------------------------
# 1. Chargement des fichiers
# ------------------------------------------------------------------
FILE_CLASSIF   = "resultats_benchmark/details_categorielle.csv"
FILE_VARIABLES = "resultats_benchmark/details_continues.csv"
FILE_RANK      = "resultats_benchmark/leaderboard_global.csv"  # à adapter

df_classif   = pd.read_csv(FILE_CLASSIF)
df_variables = pd.read_csv(FILE_VARIABLES)
df_rank      = pd.read_csv(FILE_RANK)

# ------------------------------------------------------------------
# 2. Normalisation des noms de configuration
#    -> clé canonique commune aux 3 fichiers
# ------------------------------------------------------------------
def normalize_config(cfg: str) -> str:
    """Retire les suffixes de chemin (ex: '/checkpoints') et normalise."""
    if pd.isna(cfg):
        return cfg
    return (
        str(cfg)
        .strip()
        .replace("\\", "/")
        .split("/")[0]           # garde uniquement la partie avant le premier '/'
        .lower()
    )

for df in (df_classif, df_variables, df_rank):
    df["config_key"] = df["config"].apply(normalize_config)

# ------------------------------------------------------------------
# 3. Vérification : mêmes configs dans les 3 fichiers ?
# ------------------------------------------------------------------
set_c = set(df_classif["config_key"])
set_v = set(df_variables["config_key"])
set_r = set(df_rank["config_key"])

print("=" * 70)
print("VÉRIFICATION DES CONFIGURATIONS")
print("=" * 70)
print(f"Configs classif   : {len(set_c)}")
print(f"Configs variables : {len(set_v)}")
print(f"Configs rank      : {len(set_r)}")

uniquement_classif   = set_c - set_v - set_r
uniquement_variables = set_v - set_c - set_r
uniquement_rank      = set_r - set_c - set_v

print(f"\nPrésentes seulement dans classif   : {sorted(uniquement_classif) or 'aucune'}")
print(f"Présentes seulement dans variables : {sorted(uniquement_variables) or 'aucune'}")
print(f"Présentes seulement dans rank      : {sorted(uniquement_rank) or 'aucune'}")

intersection = set_c & set_v & set_r
print(f"\nConfigs communes aux 3 fichiers    : {len(intersection)}")

# ------------------------------------------------------------------
# 4. Agrégation des variables par config
#    (moyenne des métriques par config, toutes variables confondues)
# ------------------------------------------------------------------
metric_cols = ["MSE", "MAE", "MAPE", "SMAPE", "R2", "JS", "KL", "MMD"]

df_var_agg = (
    df_variables
    .groupby("config_key")[metric_cols]
    .mean()
    .add_prefix("var_")
)

# ------------------------------------------------------------------
# 5. Fusion finale : on nettoie d'abord les colonnes en conflit
# ------------------------------------------------------------------

# Colonnes à ne garder que côté df_rank (évite les suffixes _x/_y)
cols_rank = df_rank[["config_key", "score_continuous",
                     "score_categorical", "global_rank_score", "Rang_Final"]].copy()

# df_classif : on supprime score_categorical (redondant / conflit) et config
df_classif_clean = df_classif.drop(columns=["score_categorical", "config"], errors="ignore")

df_merged = (
    df_classif_clean
    .merge(df_var_agg, on="config_key", how="inner")
    .merge(cols_rank,     on="config_key", how="inner")
)

# À ce stade, la colonne 'config' n'existe plus → on la récupère depuis df_rank
df_merged = df_merged.merge(
    df_rank[["config_key", "config"]], on="config_key", how="left"
)

# ------------------------------------------------------------------
# 6. Classement final
# ------------------------------------------------------------------
df_merged["rank_moyen_global"] = (
    df_merged["rank_global_cat"] + df_merged["Rang_Final"]
) / 2

df_merged = df_merged.sort_values("rank_moyen_global").reset_index(drop=True)

# ------------------------------------------------------------------
# 7. Affichage — on vérifie d'abord ce qui est réellement dispo
# ------------------------------------------------------------------
print("Colonnes disponibles :", list(df_merged.columns))

cols_affichage = [c for c in [
    "config",
    "accuracy_globale", "accuracy_minoritaire", "macro_f1_arith",
    "score_categorical", "global_rank_score", "Rang_Final",
    "rank_moyen_global",
    "var_MSE", "var_MAE", "var_R2", "var_JS",
] if c in df_merged.columns]

print("\n" + "=" * 70)
print("TABLEAU RÉCAPITULATIF (trié du meilleur au moins bon)")
print("=" * 70)
with pd.option_context("display.max_columns", None, "display.width", 200):
    print(df_merged[cols_affichage].to_string(index=False))

# ------------------------------------------------------------------
# 8. Top / Flop 5
# ------------------------------------------------------------------
top5  = df_merged.head(5)["config"].tolist()
flop5 = df_merged.tail(5)["config"].tolist()
print("\nTOP 5  :", top5)
print("FLOP 5 :", flop5)

# ------------------------------------------------------------------
# 9. Export
# ------------------------------------------------------------------
df_merged.to_csv("recap_configurations.csv", index=False)
print("\nFichier 'recap_configurations.csv' exporté.")

VÉRIFICATION DES CONFIGURATIONS
Configs classif   : 21
Configs variables : 21
Configs rank      : 21

Présentes seulement dans classif   : aucune
Présentes seulement dans variables : aucune
Présentes seulement dans rank      : aucune

Configs communes aux 3 fichiers    : 21
Colonnes disponibles : ['accuracy_globale', 'accuracy_minoritaire', 'macro_f1_arith', 'weighted_f1', 'fsg_macro', 'fsg_weighted', 'gmean_strict', 'gmean_smoothed', 'macro_auc_roc', 'weighted_auc_roc', 'auc_pr_macro', 'accuracy_globale_rank', 'accuracy_minoritaire_rank', 'macro_f1_arith_rank', 'weighted_f1_rank', 'fsg_macro_rank', 'fsg_weighted_rank', 'gmean_strict_rank', 'gmean_smoothed_rank', 'macro_auc_roc_rank', 'weighted_auc_roc_rank', 'auc_pr_macro_rank', 'rank_imbalanced', 'rank_global_cat', 'config_key', 'var_MSE', 'var_MAE', 'var_MAPE', 'var_SMAPE', 'var_R2', 'var_JS', 'var_KL', 'var_MMD', 'score_continuous', 'score_categorical', 'global_rank_score', 'Rang_Final', 'config', 'rank_moyen_global']

TABLEAU RÉCA